In [ ]:
import scanpy as sc
import pertpy as pt
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"]  = 42
import logging
logging.getLogger("fontTools").setLevel(logging.WARNING)


In [ ]:
adata_immune = sc.read_h5ad("/path/to/your/file.h5ad")

In [ ]:
adata_immune

In [ ]:
phase_map = {
    "uninjured": "uninjured",
    "d1": "acute",    "d2": "acute",    "d3": "acute",
    "d4": "subacute", "d5": "subacute", "d7": "subacute",
    "d9": "late",     "d14": "late",
}

adata_immune.obs["phase"] = adata_immune.obs["day"].map(phase_map).astype(str)

print(adata_immune.obs["phase"].value_counts())

In [ ]:
# Convert to plain string first (removes categorical dtype entirely)
adata_immune.obs['inj_type'] = adata_immune.obs['inj_type'].astype(str)
adata_immune.obs['phase'] = adata_immune.obs['phase'].astype(str)

# Now collapse all uninjured_* into 'uninjured'
mask_uninjured = adata_immune.obs['inj_type'].str.startswith('uninjured')
adata_immune.obs.loc[mask_uninjured, 'inj_type'] = 'uninjured'
adata_immune.obs.loc[mask_uninjured, 'phase'] = 'uninjured'

# Sanity check
print(adata_immune.obs.groupby(['inj_type', 'phase']).size())

In [ ]:
import pandas as pd

# Fix obs names first
adata_immune.obs_names_make_unique()

# Now check counts
for inj in ['liver', 'myelin']:
    print(f"\n{'='*50}")
    print(f"  {inj.upper()}")
    print(f"{'='*50}")
    subset = adata_immune[adata_immune.obs['inj_type'].isin([inj, 'uninjured'])].copy()
    ct = pd.crosstab(subset.obs['broad_immune'], subset.obs['phase'])
    print(ct)
    
    print("\n⚠️  Cell types with < 10 cells in at least one phase:")
    for col in ct.columns:
        low = ct[ct[col] < 18].index.tolist()
        if low:
            print(f"  {col}: {low}")

# Run AUGUR per injury type and compare phases v1

In [ ]:
import os

injuries = ["crush", "myelin", "liver"]

comparisons = [
    ("uninjured", "acute"),
    ("acute",     "subacute"),
    ("subacute",  "late"),
]

AUGUR_DIR = "/path/to/output/"
os.makedirs(AUGUR_DIR, exist_ok=True)

ag = pt.tl.Augur("random_forest_classifier")
results = {}

for inj in injuries:
    for (phase_a, phase_b) in comparisons:
        if phase_a == "uninjured":
            subset = adata_immune[
                (adata_immune.obs['inj_type'] == 'uninjured') |
                (
                    (adata_immune.obs['inj_type'] == inj) &
                    (adata_immune.obs['phase'] == 'acute')
                )
            ].copy()
        else:
            subset = adata_immune[
                (adata_immune.obs['inj_type'] == inj) &
                (adata_immune.obs['phase'].isin([phase_a, phase_b]))
            ].copy()
        subset.obs_names_make_unique()
        subset.obs['cell_type'] = subset.obs['broad_immune'].astype(str)
        subset.obs['label']     = subset.obs['phase'].map({phase_a: 0, phase_b: 1})
        subset.X                = subset.layers['normalized']
        if subset.n_obs < 50:
            print(f"⚠️  Skipping {inj} | {phase_a} vs {phase_b} — only {subset.n_obs} cells")
            continue
        print(f"Running: {inj} | {phase_a} vs {phase_b} | {subset.n_obs} cells")
        try:
            loaded = ag.load(subset)
            result, clf = ag.predict(
                loaded,
                subsample_size=10,
                n_threads=4,
                random_state=42,
            )
            summary = result.uns['summary_metrics'].T[['mean_augur_score']].reset_index()
            summary.columns = ['cell_type', 'mean_augur_score']
            summary['inj_type']    = inj
            summary['comparison']  = f"{phase_a}_vs_{phase_b}"
            results[(inj, f"{phase_a}_vs_{phase_b}")] = summary

            # Save immediately
            fname = f"augur_{inj}_{phase_a}_vs_{phase_b}.csv"
            summary.to_csv(os.path.join(AUGUR_DIR, fname), index=False)
            print(f"✅ Done: {inj} | {phase_a} vs {phase_b}")
        except Exception as e:
            print(f"❌ Failed {inj} | {phase_a} vs {phase_b}: {e}")

# Save all results combined
all_results = pd.concat(results.values(), ignore_index=True)
all_results.to_csv(os.path.join(AUGUR_DIR, "augur_all_results.csv"), index=False)
print(f"\nAll results saved → {AUGUR_DIR}")

In [ ]:
print("TOP RESPONDING CELL TYPE PER COMPARISON:")
print("="*55)
for (inj, comp), df in results.items():
    top = df.sort_values('mean_augur_score', ascending=False).iloc[0]
    print(f"{inj:8} | {comp:25} → {top['cell_type']:25} AUC={top['mean_augur_score']:.3f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

comp_labels = ["uninjured_vs_acute", "acute_vs_subacute", "subacute_vs_late"]
comp_titles = ["Uninjured → Acute", "Acute → Subacute", "Subacute → Late"]
colors = {'liver': '#00B96C', 'myelin': '#FBB03B'}

for comp, title in zip(comp_labels, comp_titles):
    
    df_liver  = results.get(('liver', comp))
    df_myelin = results.get(('myelin', comp))

    # Get all cell types present in either condition
    all_celltypes = sorted(set(
        (df_liver['cell_type'].tolist()  if df_liver  is not None else []) +
        (df_myelin['cell_type'].tolist() if df_myelin is not None else [])
    ))

    # Build score arrays (NaN if cell type was skipped)
    liver_scores  = []
    myelin_scores = []
    for ct in all_celltypes:
        if df_liver is not None and ct in df_liver['cell_type'].values:
            liver_scores.append(df_liver.loc[df_liver['cell_type'] == ct, 'mean_augur_score'].values[0])
        else:
            liver_scores.append(np.nan)
        if df_myelin is not None and ct in df_myelin['cell_type'].values:
            myelin_scores.append(df_myelin.loc[df_myelin['cell_type'] == ct, 'mean_augur_score'].values[0])
        else:
            myelin_scores.append(np.nan)

    # Sort by average AUC across both conditions
    avg_scores = [np.nanmean([l, m]) for l, m in zip(liver_scores, myelin_scores)]
    order = np.argsort(avg_scores)
    all_celltypes  = [all_celltypes[i]  for i in order]
    liver_scores   = [liver_scores[i]   for i in order]
    myelin_scores  = [myelin_scores[i]  for i in order]

    # Plot
    x = np.arange(len(all_celltypes))
    bar_width = 0.35

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.barh(x - bar_width/2, liver_scores,  bar_width, color=colors['liver'],  label='Liver',  alpha=0.9)
    ax.barh(x + bar_width/2, myelin_scores, bar_width, color=colors['myelin'], label='Myelin', alpha=0.9)

    ax.axvline(0.5, color='black', linestyle='--', linewidth=1, label='random (0.5)')
    ax.set_xlim(0.4, 1.0)
    ax.set_yticks(x)
    ax.set_yticklabels(all_celltypes, fontsize=11)
    ax.set_xlabel('Augur score (AUC)', fontsize=12)
    ax.set_title(f'Immune cell prioritization — {title}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)

    plt.tight_layout()
    plt.savefig(f'augur_{comp}.pdf', bbox_inches='tight', dpi=200)
    plt.show()
    print(f"✅ Saved: augur_{comp}.pdf")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

comp_labels = ["uninjured_vs_acute", "acute_vs_subacute", "subacute_vs_late"]
comp_titles = ["Uninjured → Acute", "Acute → Subacute", "Subacute → Late"]

for comp, title in zip(comp_labels, comp_titles):

    df_liver  = results.get(('liver', comp))
    df_myelin = results.get(('myelin', comp))

    # Get cell types present in both
    all_celltypes = sorted(set(
        (df_liver['cell_type'].tolist()  if df_liver  is not None else []) +
        (df_myelin['cell_type'].tolist() if df_myelin is not None else [])
    ))

    # Build score arrays
    liver_scores  = []
    myelin_scores = []
    for ct in all_celltypes:
        l = df_liver.loc[df_liver['cell_type'] == ct, 'mean_augur_score'].values[0] \
            if df_liver is not None and ct in df_liver['cell_type'].values else np.nan
        m = df_myelin.loc[df_myelin['cell_type'] == ct, 'mean_augur_score'].values[0] \
            if df_myelin is not None and ct in df_myelin['cell_type'].values else np.nan
        liver_scores.append(l)
        myelin_scores.append(m)

    # Compute log2 ratio (myelin / liver)
    # log2 > 0 = myelin driven, log2 < 0 = liver driven
    ratios = []
    for l, m in zip(liver_scores, myelin_scores):
        if np.isnan(l) or np.isnan(m) or l == 0:
            ratios.append(np.nan)
        else:
            ratios.append(np.log2(m / l))

    # Sort by ratio
    order         = np.argsort(ratios)
    all_celltypes = [all_celltypes[i] for i in order]
    ratios        = [ratios[i]        for i in order]

    # Color bars by direction
    bar_colors = ['#00B96C' if r < 0 else '#FBB03B' for r in ratios]

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(all_celltypes, ratios, color=bar_colors, alpha=0.9)
    ax.axvline(0, color='black', linestyle='--', linewidth=1)
    ax.set_xlabel('log2(Myelin AUC / Liver AUC)\n← liver driven          myelin driven →', fontsize=11)
    ax.set_title(f'Immune cell prioritization — {title}', fontsize=13, fontweight='bold')

    # Custom legend
    from matplotlib.patches import Patch
    ax.legend(handles=[
        Patch(color='#00B96C', label='Liver driven'),
        Patch(color='#FBB03B', label='Myelin driven')
    ], fontsize=10)

    plt.tight_layout()
    plt.savefig(f'augur_ratio_{comp}.pdf', bbox_inches='tight', dpi=200)
    plt.show()
    print(f"✅ Saved: augur_ratio_{comp}.pdf")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ── Cell type groups ──────────────────────────────────────────────────────────
celltype_groups = {
    "Microglia":       ["MG_High_mito", "MG_Homeostatic", "MG_Ifn", "MG_Inflammatory",
                        "MG_Lipid_metabo", "MG_Prolif", "MG_Repair", "MG_migrating"],
    "Macrophages":     ["Macro_Inflammatory", "Macro_chemotaxis", "Monocytes"],
    "Dendritic_cells": ["Dendritic_cells"],
    "T_cells":         ["T_cells"],
}

comp_labels = ["uninjured_vs_acute", "acute_vs_subacute", "subacute_vs_late"]
comp_titles = ["Uninjured → Acute", "Acute → Subacute", "Subacute → Late"]
colors = {'liver': '#E07B54', 'myelin': '#5B8DB8'}

def get_group_scores(df, celltype_groups):
    """Average augur scores per group, counting missing cell types as 0."""
    group_scores = {}
    for group, members in celltype_groups.items():
        if df is None:
            group_scores[group] = 0
            continue
        scores = []
        for ct in members:
            if df['cell_type'].isin([ct]).any():
                scores.append(df.loc[df['cell_type'] == ct, 'mean_augur_score'].values[0])
            else:
                scores.append(0)  # missing = 0
        group_scores[group] = np.mean(scores)
    return group_scores

# ── Plot — one per phase comparison ──────────────────────────────────────────
for comp, title in zip(comp_labels, comp_titles):

    df_liver  = results.get(('liver',  comp))
    df_myelin = results.get(('myelin', comp))

    liver_group_scores  = get_group_scores(df_liver,  celltype_groups)
    myelin_group_scores = get_group_scores(df_myelin, celltype_groups)

    # Sort by average AUC across both conditions
    groups        = list(celltype_groups.keys())
    liver_scores  = [liver_group_scores[g]  for g in groups]
    myelin_scores = [myelin_group_scores[g] for g in groups]
    avg           = [np.nanmean([l, m]) for l, m in zip(liver_scores, myelin_scores)]
    order         = np.argsort(avg)
    groups        = [groups[i]        for i in order]
    liver_scores  = [liver_scores[i]  for i in order]
    myelin_scores = [myelin_scores[i] for i in order]

    # Plot
    x         = np.arange(len(groups))
    bar_width = 0.35

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(x - bar_width/2, liver_scores,  bar_width, color=colors['liver'],  label='Liver',  alpha=0.9)
    ax.barh(x + bar_width/2, myelin_scores, bar_width, color=colors['myelin'], label='Myelin', alpha=0.9)

    ax.axvline(0.5, color='black', linestyle='--', linewidth=1, label='random (0.5)')
    ax.set_xlim(0.4, 1.0)
    ax.set_yticks(x)
    ax.set_yticklabels(groups, fontsize=12)
    ax.set_xlabel('Mean Augur score (AUC)', fontsize=12)
    ax.set_title(f'Immune cell prioritization — {title}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)

    plt.tight_layout()
    plt.savefig(f'augur_grouped_{comp}.pdf', bbox_inches='tight', dpi=200)
    plt.show()
    print(f"✅ Saved: augur_grouped_{comp}.pdf")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

celltype_groups = {
    "Microglia":       ["MG_High_mito", "MG_Homeostatic", "MG_Ifn", "MG_Inflammatory",
                        "MG_Lipid_metabo", "MG_Prolif", "MG_Repair", "MG_migrating"],
    "Macrophages":     ["Macro_Inflammatory", "Macro_chemotaxis", "Monocytes"],
    "Dendritic_cells": ["Dendritic_cells"],
    "T_cells":         ["T_cells"],
}

comp_labels = ["uninjured_vs_acute", "acute_vs_subacute", "subacute_vs_late"]
comp_titles = ["Uninjured → Acute", "Acute → Subacute", "Subacute → Late"]
colors = {'liver': '#E07B54', 'myelin': '#5B8DB8'}

def get_group_scores(df, celltype_groups):
    """Average augur scores per group, ignoring missing cell types."""
    group_scores = {}
    for group, members in celltype_groups.items():
        if df is None:
            group_scores[group] = np.nan
            continue
        present = df[df['cell_type'].isin(members)]
        group_scores[group] = present['mean_augur_score'].mean() if len(present) > 0 else np.nan
    return group_scores

for comp, title in zip(comp_labels, comp_titles):

    df_liver  = results.get(('liver',  comp))
    df_myelin = results.get(('myelin', comp))

    liver_group_scores  = get_group_scores(df_liver,  celltype_groups)
    myelin_group_scores = get_group_scores(df_myelin, celltype_groups)

    # Sort by average AUC
    groups = list(celltype_groups.keys())
    liver_scores  = [liver_group_scores[g]  for g in groups]
    myelin_scores = [myelin_group_scores[g] for g in groups]
    avg = [np.nanmean([l, m]) for l, m in zip(liver_scores, myelin_scores)]
    order = np.argsort(avg)
    groups        = [groups[i]        for i in order]
    liver_scores  = [liver_scores[i]  for i in order]
    myelin_scores = [myelin_scores[i] for i in order]

    # Plot
    x = np.arange(len(groups))
    bar_width = 0.35

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(x - bar_width/2, liver_scores,  bar_width, color=colors['liver'],  label='Liver',  alpha=0.9)
    ax.barh(x + bar_width/2, myelin_scores, bar_width, color=colors['myelin'], label='Myelin', alpha=0.9)

    ax.axvline(0.5, color='black', linestyle='--', linewidth=1, label='random (0.5)')
    ax.set_xlim(0.4, 1.0)
    ax.set_yticks(x)
    ax.set_yticklabels(groups, fontsize=12)
    ax.set_xlabel('Mean Augur score (AUC)', fontsize=12)
    ax.set_title(f'Immune cell prioritization — {title}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)

    plt.tight_layout()
    plt.savefig(f'augur_grouped_{comp}.pdf', bbox_inches='tight', dpi=200)
    plt.show()
    print(f"✅ Saved: augur_grouped_{comp}.pdf")

# Run AUGUR per phase and per celltype (broad) and compared injury type v1

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ── 1. Run Augur — liver vs myelin per phase ──────────────────────────────────
ag = pt.tl.Augur("random_forest_classifier")
results_lvm = {}  # lvm = liver vs myelin

phases = ['acute', 'subacute', 'late']

for phase in phases:

    subset = adata_augur[
        (adata_augur.obs['inj_type'].isin(['liver', 'myelin'])) &
        (adata_augur.obs['phase'] == phase)
    ].copy()

    subset.obs_names_make_unique()
    subset.obs['cell_type'] = subset.obs['predicted_celltype'].astype(str)
    subset.obs['label']     = subset.obs['inj_type'].map({'liver': 0, 'myelin': 1})
    subset.X                = subset.layers['normalized']

    if subset.n_obs < 50:
        print(f"⚠️  Skipping {phase} — only {subset.n_obs} cells")
        continue

    print(f"Running: liver vs myelin | {phase} | {subset.n_obs} cells")
    print(f"  Label counts: {subset.obs['label'].value_counts().to_dict()}")

    try:
        loaded = ag.load(subset)
        result, clf = ag.predict(
            loaded,
            subsample_size=10,
            n_threads=4,
            random_state=42,
        )
        summary = result.uns['summary_metrics'].T[['mean_augur_score']].reset_index()
        summary.columns = ['cell_type', 'mean_augur_score']
        results_lvm[phase] = summary

        # Save immediately
        summary.to_csv(f"augur_liver_vs_myelin_{phase}.csv", index=False)
        print(f"✅ Done: {phase}")

    except Exception as e:
        print(f"❌ Failed {phase}: {e}")

# ── 2. Aggregate per cell group ───────────────────────────────────────────────
celltype_groups = {
    "Microglia":       ["MG_High_mito", "MG_Homeostatic", "MG_Ifn", "MG_Inflammatory",
                        "MG_Lipid_metabo", "MG_Prolif", "MG_Repair", "MG_migrating"],
    "Macrophages":     ["Macro_Inflammatory", "Macro_chemotaxis", "Monocytes"],
    "Dendritic_cells": ["Dendritic_cells"],
    "T_cells":         ["T_cells"],
}

def get_group_scores(df, celltype_groups):
    group_scores = {}
    for group, members in celltype_groups.items():
        if df is None:
            group_scores[group] = 0
            continue
        scores = []
        for ct in members:
            if df['cell_type'].isin([ct]).any():
                scores.append(df.loc[df['cell_type'] == ct, 'mean_augur_score'].values[0])
            else:
                scores.append(0)
        group_scores[group] = np.mean(scores)
    return group_scores

# ── 3. Plot — one per phase ───────────────────────────────────────────────────
phase_titles = {
    'acute':    'Acute phase',
    'subacute': 'Subacute phase',
    'late':     'Late phase'
}

for phase in phases:
    df = results_lvm.get(phase)
    if df is None:
        continue

    group_scores = get_group_scores(df, celltype_groups)

    # Sort by AUC
    groups = list(group_scores.keys())
    scores = [group_scores[g] for g in groups]
    order  = np.argsort(scores)
    groups = [groups[i] for i in order]
    scores = [scores[i] for i in order]

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(groups, scores, color='#7B61C4', alpha=0.9)
    ax.axvline(0.5, color='black', linestyle='--', linewidth=1, label='random (0.5)')
    ax.set_xlim(0.4, 1.0)
    ax.set_xlabel('Augur score (AUC)\n← liver driven         myelin driven →', fontsize=11)
    ax.set_title(f'Liver vs Myelin — {phase_titles[phase]}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)

    plt.tight_layout()
    plt.savefig(f'augur_liver_vs_myelin_{phase}.pdf', bbox_inches='tight', dpi=200)
    plt.show()
    print(f"✅ Saved: augur_liver_vs_myelin_{phase}.pdf")

In [ ]:
phase_titles = {
    'acute':    'Acute phase',
    'subacute': 'Subacute phase',
    'late':     'Late phase'
}

for phase in phases:
    df = results_lvm.get(phase)
    if df is None:
        continue

    # Sort by AUC
    df_sorted = df.sort_values('mean_augur_score', ascending=True)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(df_sorted['cell_type'], df_sorted['mean_augur_score'], color='#7B61C4', alpha=0.9)
    ax.axvline(0.5, color='black', linestyle='--', linewidth=1, label='random (0.5)')
    ax.set_xlim(0, 1.0)
    ax.set_xlabel('Augur score (AUC)\n← liver driven         myelin driven →', fontsize=11)
    ax.set_title(f'Liver vs Myelin — {phase_titles[phase]}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)

    plt.tight_layout()
    plt.savefig(f'augur_liver_vs_myelin_celltype_{phase}.pdf', bbox_inches='tight', dpi=200)
    plt.show()
    print(f"✅ Saved: augur_liver_vs_myelin_celltype_{phase}.pdf")

# Run AUGUR and compared injur model per phase v2

In [ ]:
AUGUR_DIR = "/path/to/output/"
os.makedirs(AUGUR_DIR, exist_ok=True)

# Phases to compare myelin vs liver at each phase
phases = ["acute", "subacute", "late"]

ag      = pt.tl.Augur("random_forest_classifier")
results = {}

for phase in phases:
    # Subset to myelin + liver at this phase
    subset = adata_immune[
        (adata_immune.obs['inj_type'].isin(["myelin", "liver"])) &
        (adata_immune.obs['phase'] == phase)
    ].copy()

    if subset.n_obs < 18:
        print(f"⚠️  Skipping {phase} — only {subset.n_obs} cells")
        continue

    subset.obs_names_make_unique()
    subset.obs['cell_type'] = subset.obs['broad_immune'].astype(str)
    subset.obs['label']     = subset.obs['inj_type'].map({"myelin": 0, "liver": 1})
    subset.X                = subset.layers['transformed']

    print(f"Running: myelin vs liver | {phase} | {subset.n_obs} cells")
    print(f"  myelin: {(subset.obs['inj_type']=='myelin').sum():,} | "
          f"liver: {(subset.obs['inj_type']=='liver').sum():,}")

    try:
        loaded       = ag.load(subset)
        result, clf  = ag.predict(
            loaded,
            subsample_size = 18,
            n_threads      = 4,
            random_state   = 42,
            span = 0.5,
        )

        summary = result.uns['summary_metrics'].T[['mean_augur_score']].reset_index()
        summary.columns   = ['cell_type', 'mean_augur_score']
        summary['phase']  = phase
        summary['comparison'] = "myelin_vs_liver"
        results[phase]    = summary

        fname = f"augur_myelin_vs_liver_{phase}.csv"
        summary.to_csv(os.path.join(AUGUR_DIR, fname), index=False)
        print(f"✅ Done: myelin vs liver | {phase}")
        print(summary.sort_values("mean_augur_score", ascending=False).to_string())

    except Exception as e:
        print(f"❌ Failed {phase}: {e}")

# Save all combined
if results:
    all_results = pd.concat(results.values(), ignore_index=True)
    all_results.to_csv(os.path.join(AUGUR_DIR, "augur_myelin_vs_liver_all.csv"), index=False)
    print(f"\nAll results saved → {AUGUR_DIR}")

In [ ]:
AUGUR_DIR = "/path/to/output/"
os.makedirs(AUGUR_DIR, exist_ok=True)

# Phases to compare myelin vs liver at each phase
phases = ["acute", "subacute", "late"]

ag      = pt.tl.Augur("random_forest_classifier")
results = {}

for phase in phases:
    # Subset to myelin + liver at this phase
    subset = adata_immune[
        (adata_immune.obs['inj_type'].isin(["myelin", "liver"])) &
        (adata_immune.obs['phase'] == phase)
    ].copy()

    if subset.n_obs < 18:
        print(f"⚠️  Skipping {phase} — only {subset.n_obs} cells")
        continue

    subset.obs_names_make_unique()
    subset.obs['cell_type'] = subset.obs['immune_subset'].astype(str)
    subset.obs['label']     = subset.obs['inj_type'].map({"myelin": 0, "liver": 1})
    subset.X                = subset.layers['transformed']

    print(f"Running: myelin vs liver | {phase} | {subset.n_obs} cells")
    print(f"  myelin: {(subset.obs['inj_type']=='myelin').sum():,} | "
          f"liver: {(subset.obs['inj_type']=='liver').sum():,}")

    try:
        loaded       = ag.load(subset)
        result, clf  = ag.predict(
            loaded,
            subsample_size = 10,
            n_threads      = 4,
            random_state   = 42,
            span = 0.5,
            min_cells = 18,
            
        )

        summary = result.uns['summary_metrics'].T[['mean_augur_score']].reset_index()
        summary.columns   = ['cell_type', 'mean_augur_score']
        summary['phase']  = phase
        summary['comparison'] = "myelin_vs_liver"
        results[phase]    = summary

        fname = f"augur_myelin_vs_liver_{phase}.csv"
        summary.to_csv(os.path.join(AUGUR_DIR, fname), index=False)
        print(f"✅ Done: myelin vs liver | {phase}")
        print(summary.sort_values("mean_augur_score", ascending=False).to_string())

    except Exception as e:
        print(f"❌ Failed {phase}: {e}")

# Save all combined
if results:
    all_results = pd.concat(results.values(), ignore_index=True)
    all_results.to_csv(os.path.join(AUGUR_DIR, "augur_myelin_vs_liver_all.csv"), index=False)
    print(f"\nAll results saved → {AUGUR_DIR}")

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
x_min, x_max = 0.45, 0.7   # ← adjust x scale here

color_map_augur = {
    'DC_T_cells'                : "#87C254",
    'Macro_Lipid_handling'      : "#FFC80A",
    'Macro_Il33'                : "#FF9500",
    'Macro_Cholesterol_handling': "#CC2200",
    'Macro_ECM_secreting'       : "#97000A",
    "MG_Homeostatic"          : "#C5DCFB",
    "MG_Migrating"            : "#4AA2AA",
    "MG_Inflammatory"         : "#1A9EFF",
    "MG_Lipid_handling"       : "#17355A",
    "MG_Cholesterol_handling" : "#0045FF",
    
}

cell_order = list(color_map_augur.keys())

# ── Load results ──────────────────────────────────────────────────────────────
if not results:
    all_results = pd.read_csv(os.path.join(AUGUR_DIR, "augur_myelin_vs_liver_all.csv"))
else:
    all_results = pd.concat(results.values(), ignore_index=True)

# ── Plot ──────────────────────────────────────────────────────────────────────
phases_found = ["acute", "subacute", "late"]
phases_found = [p for p in phases_found if p in all_results["phase"].unique()]

fig, axes = plt.subplots(
    1, len(phases_found),
    figsize=(5 * len(phases_found), 5),
    sharey=False
)
if len(phases_found) == 1:
    axes = [axes]

for ax, phase in zip(axes, phases_found):
    data = all_results[all_results["phase"] == phase].copy()

    # Apply fixed order — only keep cell types present in results
    order = [ct for ct in cell_order if ct in data["cell_type"].values]
    data  = data.set_index("cell_type").reindex(order).reset_index()
    colors = [color_map_augur[ct] for ct in data["cell_type"]]

    ax.barh(
        data["cell_type"],
        data["mean_augur_score"],
        color     = colors,
        edgecolor = "white",
        linewidth = 0.5,
    )
    ax.axvline(0.5, color="black", linewidth=1, linestyle="--", alpha=0.7)
    ax.set_xlim(x_min, x_max)
    ax.set_xlabel("Mean Augur score (AUC)", fontsize=10)
    ax.set_title(phase, fontsize=12)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)    
    ax.invert_yaxis()

axes[0].set_ylabel("Cell type", fontsize=10)
plt.suptitle("Augur scores — myelin vs liver", fontsize=14)
plt.tight_layout()
plt.savefig(
    os.path.join(AUGUR_DIR, "augur_myelin_vs_liver_barplot.pdf"),
    bbox_inches="tight", dpi=300, format="pdf"
)
plt.show()

In [ ]:
import os
import subprocess
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

proj_dir = "/path/to/output/"

# ══════════════════════════════════════════════════════════════════════════════
# PARAMETERS — change only this block
# ══════════════════════════════════════════════════════════════════════════════
celltype_value       = "Macrophages"
inj_types            = ["myelin", "liver"]
phase_to_test        = "subacute"
use_day_covariate    = True
use_injury_covariate = False
MIN_CELLS_PER_REPLICATE = 10

use_padj      = False
pval_thresh   = 0.05
lfc_thresh    = 0.58
top_n_labels  = 10

OUT_DIR  = f"{proj_dir}DEG_spatial_raw_Macro_Mono/"
RSCRIPT  = "/usr/local/bin/Rscript"

COVARIATE_R_THRESHOLD            = 0.75
MIN_REPS_PER_GROUP_FOR_COVARIATE = 3
# ══════════════════════════════════════════════════════════════════════════════

analysis_label = (f"spatial_raw_{celltype_value}_"
                  f"{inj_types[0]}_vs_{inj_types[1]}_{phase_to_test}")
os.makedirs(OUT_DIR, exist_ok=True)

injury_col = "inj_type"
time_col   = "day"
sample_col = "manual_anno"
rep_col    = "replicate"

celltype_groups = {
    "Microglia":       ['MG_Homeostatic', 'MG_Activated', 'MG_Reactive', 'MG_Serpina', 'MG_Chronic'],
    "Macrophages":     ['Macro_early', 'Macro_middle', 'Macro_late', 'Macro_chronic', "Monocytes", "Dendritic_cells"],
}

phase_map = {
    "d1": "acute",    "d2": "acute",    "d3": "acute",
    "d4": "subacute", "d5": "subacute", "d7": "subacute",
    "d9": "late",     "d14": "late",
}

replicate_map_Macro = {
    # ── ACUTE ─────────────────────────────────────────────────────────────────
    # liver
    "liver_d1_1":  "rep_1",   "liver_d1_7":  "rep_2",   "liver_d1_6":  "rep_3",
    "liver_d1_5":  "rep_4",   "liver_d1_9":  "rep_4",   "liver_d1_8":  "rep_6",
    "liver_d1_4":  "rep_7",   "liver_d1_3":  "rep_8",   "liver_d1_2":  "rep_9",
    "liver_d2_4":  "rep_10",  "liver_d2_5":  "rep_11",  "liver_d2_2":  "rep_12",
    "liver_d2_3":  "rep_13",  "liver_d2_1":  "rep_11",  "liver_d2_6":  "rep_11",
    "liver_d2_7":  "rep_11",  "liver_d3_1":  "rep_17",  "liver_d3_6":  "rep_18",
    "liver_d3_7":  "rep_19",  "liver_d3_4":  "rep_19",  "liver_d3_8":  "rep_21",
    "liver_d3_5":  "rep_21",  "liver_d3_2":  "rep_23",  "liver_d3_3":  "rep_24",
    # myelin
    "myelin_d1_7": "rep_25",  "myelin_d1_6": "rep_25",  "myelin_d1_1": "rep_27",
    "myelin_d1_3": "rep_28",  "myelin_d1_2": "rep_25",  "myelin_d1_9": "rep_30",
    "myelin_d1_8": "rep_30",  "myelin_d1_4": "rep_30",  "myelin_d2_2": "rep_30",
    "myelin_d2_3": "rep_30",  "myelin_d2_4": "rep_35",  "myelin_d2_5": "rep_36",
    "myelin_d2_6": "rep_37",  "myelin_d2_7": "rep_38",  "myelin_d2_1": "rep_38",
    "myelin_d3_1": "rep_47",  "myelin_d3_2": "rep_41",  "myelin_d3_3": "rep_43",
    "myelin_d3_4": "rep_43",
    # ── SUBACUTE ──────────────────────────────────────────────────────────────
    # liver
    "liver_d4_2":  "rep_45",  "liver_d4_3":  "rep_45",  "liver_d4_8":  "rep_45",
    "liver_d4_9":  "rep_45",  "liver_d4_5":  "rep_45",  "liver_d4_6":  "rep_45",
    "liver_d4_7":  "rep_45",  "liver_d4_10": "rep_45",  "liver_d4_1":  "rep_45",
    "liver_d4_11": "rep_45",  "liver_d5_6":  "rep_55",  "liver_d5_7":  "rep_45",
    "liver_d5_2":  "rep_45",  "liver_d5_3":  "rep_58",  "liver_d5_8":  "rep_59",
    "liver_d5_4":  "rep_60",  "liver_d5_5":  "rep_61",  "liver_d5_10": "rep_62",
    "liver_d7_18": "rep_63",  "liver_d7_14": "rep_62",  "liver_d7_15": "rep_62",
    "liver_d7_19": "rep_66",  "liver_d7_12": "rep_62",  "liver_d7_13": "rep_62",
    "liver_d7_10": "rep_69",  "liver_d7_11": "rep_62",  "liver_d7_9":  "rep_71",
    "liver_d7_16": "rep_72",  "liver_d7_17": "rep_73",
    # myelin
    "myelin_d4_4":  "rep_74", "myelin_d4_14": "rep_75", "myelin_d4_8":  "rep_76",
    "myelin_d4_9":  "rep_77", "myelin_d4_5":  "rep_78", "myelin_d4_2":  "rep_79",
    "myelin_d4_3":  "rep_80", "myelin_d4_1":  "rep_81", "myelin_d4_6":  "rep_82",
    "myelin_d4_7":  "rep_83", "myelin_d5_1":  "rep_84", "myelin_d5_6":  "rep_85",
    "myelin_d5_7":  "rep_86", "myelin_d5_5":  "rep_87", "myelin_d5_2":  "rep_88",
    "myelin_d5_3":  "rep_88", "myelin_d7_12": "rep_90", "myelin_d7_13": "rep_91",
    "myelin_d7_14": "rep_92", "myelin_d7_9":  "rep_93", "myelin_d7_8":  "rep_93",
    "myelin_d7_10": "rep_93", "myelin_d7_11": "rep_96",
    # ── LATE ──────────────────────────────────────────────────────────────────
    # liver
    "liver_d9_10":  "rep_98",  "liver_d9_11":  "rep_99",  "liver_d9_8":   "rep_100",
    "liver_d9_9":   "rep_101", "liver_d14_1":  "rep_102", "liver_d14_13": "rep_103",
    "liver_d14_12": "rep_104", "liver_d14_11": "rep_105", "liver_d14_9":  "rep_106",
    "liver_d14_10": "rep_107", "liver_d14_2":  "rep_108", "liver_d14_3":  "rep_109",
    # myelin
    "myelin_d9_1":   "rep_110", "myelin_d9_6":   "rep_111", "myelin_d9_7":   "rep_112",
    "myelin_d9_8":   "rep_113", "myelin_d9_4":   "rep_114", "myelin_d9_5":   "rep_115",
    "myelin_d9_9":   "rep_116", "myelin_d9_2":   "rep_117", "myelin_d14_10": "rep_118",
    "myelin_d14_7":  "rep_119", "myelin_d14_11": "rep_120", "myelin_d14_1":  "rep_121",
    "myelin_d14_2":  "rep_122", "myelin_d14_3":  "rep_123", "myelin_d14_12": "rep_124",
    "myelin_d14_8":  "rep_125", "myelin_d14_9":  "rep_126", "myelin_d14_13": "rep_127",
}

# ── Verify ────────────────────────────────────────────────────────────────────
print(f"Total batches in map: {len(replicate_map_Macro)}")
print(f"Unique replicates:    {len(set(replicate_map_Macro.values()))}")

# ══════════════════════════════════════════════════════════════════════════════
# 1. Prepare metadata
# ══════════════════════════════════════════════════════════════════════════════
adata_immune.obs[injury_col] = adata_immune.obs[sample_col].str.split("_d").str[0]
adata_immune.obs[time_col]   = "d" + adata_immune.obs[sample_col].str.split("_d").str[1].str.split("_").str[0]
adata_immune.obs["phase"]    = adata_immune.obs[time_col].map(phase_map)

# ══════════════════════════════════════════════════════════════════════════════
# 2. Subset
# ══════════════════════════════════════════════════════════════════════════════
members       = celltype_groups[celltype_value]
celltype_mask = adata_immune.obs["immune_subset"].isin(members)
injury_mask   = adata_immune.obs[injury_col].isin(inj_types)
phase_mask    = adata_immune.obs["phase"] == phase_to_test

adata_deg = adata_immune[celltype_mask & injury_mask & phase_mask].copy()
print(f"Cells after subsetting: {adata_deg.n_obs}")

if adata_deg.n_obs == 0:
    raise ValueError(f"No cells found for {celltype_value}, {inj_types}, {phase_to_test}")

# ══════════════════════════════════════════════════════════════════════════════
# 3. Extract raw counts from layers["counts"] ← KEY DIFFERENCE
# ══════════════════════════════════════════════════════════════════════════════
X = adata_deg.layers["counts"]  # ← raw integer counts
if hasattr(X, "toarray"):
    X = X.toarray()
else:
    X = np.asarray(X)

X_scaled = np.round(X).astype(int)  # already integers, just ensure dtype
adata_deg.obs["total_counts"] = X_scaled.sum(axis=1)

print(f"\nRaw counts check (layers['counts']):")
print(f"  min:        {X_scaled.min()}")
print(f"  max:        {X_scaled.max()}")
print(f"  mean:       {X_scaled.mean():.4f}")
print(f"  % zeros:    {(X_scaled == 0).mean() * 100:.1f}%")
print(f"  % integers: {(X_scaled == np.round(X_scaled)).mean() * 100:.1f}%")
print("✅ Using raw counts from layers['counts']")

# ══════════════════════════════════════════════════════════════════════════════
# 4. Apply replicate map
# ══════════════════════════════════════════════════════════════════════════════
adata_deg.obs[rep_col] = adata_deg.obs[sample_col].map(replicate_map_Macro)

unmapped = adata_deg.obs.loc[adata_deg.obs[rep_col].isna(), sample_col].unique()
if len(unmapped) > 0:
    print(f"\n⚠ Unmapped batches (will be dropped): {sorted(unmapped)}")
    keep_mask = adata_deg.obs[rep_col].notna().values
    adata_deg = adata_deg[keep_mask].copy()
    X_scaled  = X_scaled[keep_mask]
else:
    print(f"\n✅ All batches mapped successfully")

print(f"Unique replicates: {adata_deg.obs[rep_col].nunique()}")

# ══════════════════════════════════════════════════════════════════════════════
# 5. Pseudobulk
# ══════════════════════════════════════════════════════════════════════════════
def make_pseudobulk(adata, X_scaled, rep_col):
    replicates  = adata.obs[rep_col].astype(str).values
    unique_reps = pd.Index(pd.unique(replicates))

    pb_rows = []
    for r in unique_reps:
        idx    = np.where(replicates == r)[0]
        summed = X_scaled[idx].sum(axis=0)
        pb_rows.append(summed)

    pb_counts = pd.DataFrame(
        pb_rows,
        index=unique_reps,
        columns=adata.var_names,
        dtype=int,
    )

    pb_meta = (
        adata.obs
        .groupby(rep_col, observed=True)
        [[injury_col, "phase", time_col]]
        .agg("first")
        .copy()
    )
    pb_meta["n_cells"]     = adata.obs.groupby(rep_col).size()
    pb_meta["mean_counts"] = adata.obs.groupby(rep_col)["total_counts"].mean().round(4)

    return pb_counts, pb_meta

pb_counts, pb_meta = make_pseudobulk(adata_deg, X_scaled, rep_col)

print(f"\npb_counts shape: {pb_counts.shape}")
print(f"\nCell counts per replicate:")
print(pb_meta[[injury_col, time_col, "n_cells"]]
      .sort_values([injury_col, time_col, "n_cells"])
      .to_string())

print(f"\nReplicates below {MIN_CELLS_PER_REPLICATE} cells:")
low = pb_meta[pb_meta["n_cells"] < MIN_CELLS_PER_REPLICATE]
print(low[[injury_col, time_col, "n_cells"]].to_string() if len(low) > 0 else "None ✅")

print(f"\nDay balance per condition:")
print(pb_meta.groupby([injury_col, time_col]).size().unstack(fill_value=0))

print(f"\nSummary per condition:")
print(pb_meta.groupby(injury_col)["n_cells"]
      .agg(["count", "sum", "min", "max", "mean"])
      .rename(columns={"count": "n_reps", "sum": "total_cells", "mean": "mean_cells"})
      .round(1))

# ══════════════════════════════════════════════════════════════════════════════
# 6. QC plot
# ══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(data=adata_deg.obs, x=injury_col, y="total_counts",
            palette={inj_types[0]: "steelblue", inj_types[1]: "orange"},
            order=inj_types, ax=axes[0], showfliers=False)
sns.stripplot(data=adata_deg.obs, x=injury_col, y="total_counts",
              palette={inj_types[0]: "steelblue", inj_types[1]: "orange"},
              order=inj_types, ax=axes[0], size=2, alpha=0.3, jitter=True)
axes[0].set_title("Raw counts per cell (layers['counts'])")
axes[0].set_xlabel("")

day_counts = pb_meta.groupby([injury_col, time_col]).size().unstack(fill_value=0)
day_counts.T.plot(kind="bar", ax=axes[1],
                  color={inj_types[0]: "steelblue", inj_types[1]: "orange"})
axes[1].set_title("Replicates per day per condition")
axes[1].set_xlabel("Day")
axes[1].set_ylabel("N replicates")
axes[1].tick_params(axis="x", rotation=0)

plt.suptitle(f"{celltype_value} — {phase_to_test} | "
             f"{inj_types[0]} vs {inj_types[1]} (raw spatial counts)")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}QC_{analysis_label}.pdf", bbox_inches="tight")
plt.show()

# ══════════════════════════════════════════════════════════════════════════════
# 7. Covariate decision
# ══════════════════════════════════════════════════════════════════════════════
def decide_covariate_use(meta_ph, inj_types):
    counts_by_group = meta_ph.groupby(injury_col).size().to_dict()
    n_0 = int(counts_by_group.get(inj_types[0], 0))
    n_1 = int(counts_by_group.get(inj_types[1], 0))

    injury_binary = (meta_ph[injury_col] == inj_types[0]).astype(int)

    if meta_ph["mean_counts"].nunique() <= 1:
        r, p = np.nan, np.nan
    else:
        r, p = stats.pointbiserialr(injury_binary, meta_ph["mean_counts"])

    enough_reps      = (n_0 >= MIN_REPS_PER_GROUP_FOR_COVARIATE) and \
                       (n_1 >= MIN_REPS_PER_GROUP_FOR_COVARIATE)
    low_collinearity = pd.notna(r) and (abs(r) <= COVARIATE_R_THRESHOLD)
    use_covariate    = bool(enough_reps and low_collinearity)

    print(f"\nCorrelation injury ~ mean_counts: r={r:.3f}, p={p:.3f}")
    print(f"n_{inj_types[0]}={n_0}, n_{inj_types[1]}={n_1}")
    if not enough_reps:
        print(f"  → DO NOT use mean_counts covariate: too few replicates")
    elif pd.isna(r):
        print(f"  → DO NOT use mean_counts covariate: no variation")
    elif abs(r) > COVARIATE_R_THRESHOLD:
        print(f"  → DO NOT use mean_counts covariate: collinearity too high (|r|={abs(r):.3f})")
    else:
        print(f"  → USE mean_counts covariate (|r|={abs(r):.3f})")
    print(f"  → Day covariate:    {'YES' if use_day_covariate else 'NO'}")
    print(f"  → Injury covariate: {'YES' if use_injury_covariate else 'NO'}")

    return use_covariate

use_mc_covariate = decide_covariate_use(pb_meta, inj_types)

terms = []
if use_day_covariate:    terms.append("day")
if use_mc_covariate:     terms.append("mc_scaled")
if use_injury_covariate: terms.append("injury_type")
terms.append("injury")
design_str = "~ " + " + ".join(terms)
print(f"\nFinal design: {design_str}")

# ══════════════════════════════════════════════════════════════════════════════
# 8. edgeR R script
# ══════════════════════════════════════════════════════════════════════════════
R_SCRIPT_TEMPLATE = r"""
suppressPackageStartupMessages({{ library(edgeR) }})

counts_path          <- "{counts_path}"
meta_path            <- "{meta_path}"
out_path             <- "{out_path}"
use_mc_covariate     <- {use_mc_covariate}
use_day_covariate    <- {use_day_covariate}
use_injury_covariate <- {use_injury_covariate}
ref_level            <- "{ref_level}"
test_level           <- "{test_level}"

counts_df <- read.csv(counts_path, row.names = 1, check.names = FALSE)
counts    <- t(as.matrix(counts_df))
meta      <- read.csv(meta_path, row.names = 1, check.names = FALSE)
meta      <- meta[colnames(counts), , drop = FALSE]

injury      <- factor(meta$inj_type, levels = c(ref_level, test_level))
day         <- factor(meta$day)
injury_type <- factor(meta$inj_type)

terms <- c()
if (use_day_covariate)    terms <- c(terms, "day")
if (use_mc_covariate) {{
    mc_scaled <- meta$mean_counts_scaled
    terms <- c(terms, "mc_scaled")
}}
if (use_injury_covariate) terms <- c(terms, "injury_type")
terms <- c(terms, "injury")

formula_str <- paste("~", paste(terms, collapse = " + "))
cat("Formula:", formula_str, "\n")

design <- model.matrix(as.formula(formula_str))
rownames(design) <- colnames(counts)
cat("Design:", paste(colnames(design), collapse=", "), "\n")
cat("Design dimensions:", nrow(design), "x", ncol(design), "\n")

dge  <- DGEList(counts = counts, group = injury)
keep <- rowSums(dge$counts) >= 5
cat("Genes passing rowSums filter:", sum(keep), "\n")
dge  <- dge[keep, , keep.lib.sizes = FALSE]
dge  <- calcNormFactors(dge, method = "TMM")
dge  <- estimateDisp(dge, design, robust = TRUE)
fit  <- glmQLFit(dge, design, robust = TRUE)
qlf  <- glmQLFTest(fit, coef = ncol(design))

result <- topTags(qlf, n = Inf, sort.by = "PValue")$table
colnames(result)[colnames(result) == "logFC"]  <- "log2FoldChange"
colnames(result)[colnames(result) == "PValue"] <- "pvalue"
colnames(result)[colnames(result) == "FDR"]    <- "padj"
result$gene <- rownames(result)
result <- result[, c("gene", "log2FoldChange", "logCPM", "F", "pvalue", "padj")]
write.csv(result, out_path, row.names = FALSE)
cat("edgeR done:", nrow(result), "genes tested\n")
"""

# ══════════════════════════════════════════════════════════════════════════════
# 9. Run edgeR
# ══════════════════════════════════════════════════════════════════════════════
def run_edgeR(pb_counts, pb_meta, inj_types, use_mc_covariate,
              use_day_covariate, use_injury_covariate,
              min_cells=MIN_CELLS_PER_REPLICATE):

    meta_ph   = pb_meta.copy()
    counts_ph = pb_counts.loc[meta_ph.index].copy()

    valid   = meta_ph["n_cells"] >= min_cells
    dropped = meta_ph[~valid].index.tolist()
    if dropped:
        print(f"⚠ Dropping {len(dropped)} replicate(s) with <{min_cells} cells: {dropped}")
    meta_ph   = meta_ph[valid]
    counts_ph = counts_ph.loc[meta_ph.index]

    conditions = meta_ph[injury_col].unique()
    if not set(inj_types).issubset(set(conditions)):
        raise ValueError(f"Lost a condition after filtering. Found: {conditions}\n"
                         f"Try reducing MIN_CELLS_PER_REPLICATE (currently {min_cells})")

    keep      = (counts_ph > 0).sum(axis=0) >= 2
    counts_ph = counts_ph.loc[:, keep]
    print(f"Replicates: {len(meta_ph)} | Genes after pre-filter: {keep.sum()}")
    print(f"Day balance:\n"
          f"{meta_ph.groupby([injury_col, time_col]).size().unstack(fill_value=0)}")

    meta_ph = meta_ph.copy()
    meta_ph["mean_counts_scaled"] = meta_ph["mean_counts"] / meta_ph["mean_counts"].mean()

    with tempfile.TemporaryDirectory() as tmpdir:
        counts_path = os.path.join(tmpdir, "counts.csv")
        meta_path   = os.path.join(tmpdir, "meta.csv")
        out_path    = os.path.join(tmpdir, "result.csv")
        r_path      = os.path.join(tmpdir, "run_edgeR.R")

        counts_ph.to_csv(counts_path)
        meta_ph[[injury_col, time_col, "mean_counts", "mean_counts_scaled"]].to_csv(meta_path)

        r_script = R_SCRIPT_TEMPLATE.format(
            counts_path=counts_path,
            meta_path=meta_path,
            out_path=out_path,
            use_mc_covariate="TRUE" if use_mc_covariate else "FALSE",
            use_day_covariate="TRUE" if use_day_covariate else "FALSE",
            use_injury_covariate="TRUE" if use_injury_covariate else "FALSE",
            ref_level=inj_types[0],
            test_level=inj_types[1],
        )
        with open(r_path, "w") as f:
            f.write(r_script)

        proc = subprocess.run([RSCRIPT, "--vanilla", r_path],
                              capture_output=True, text=True)

        if proc.returncode != 0:
            print(f"R stdout:\n{proc.stdout}")
            print(f"R stderr:\n{proc.stderr}")
            raise RuntimeError("edgeR failed")

        print(proc.stdout.strip())
        res = pd.read_csv(out_path, index_col="gene")

    return res.sort_values("pvalue")

res = run_edgeR(pb_counts, pb_meta, inj_types, use_mc_covariate,
                use_day_covariate, use_injury_covariate)
out_csv = f"{OUT_DIR}DEG_{analysis_label}.csv"
res.to_csv(out_csv)

# ══════════════════════════════════════════════════════════════════════════════
# 10. Diagnostic
# ══════════════════════════════════════════════════════════════════════════════
pval_col = "padj" if use_padj else "pvalue"
n_sig = ((res[pval_col] < pval_thresh) & (res["log2FoldChange"].abs() > lfc_thresh)).sum()

print(f"\nResults saved → {out_csv}")
print(f"Significant genes ({pval_col}<{pval_thresh}, |lfc|>{lfc_thresh}): {n_sig}")
print(f"\nGenes with pvalue < 0.05: {(res['pvalue'] < 0.05).sum()}")
print(f"Genes with pvalue < 0.01: {(res['pvalue'] < 0.01).sum()}")
print(f"Genes with padj  < 0.05: {(res['padj']  < 0.05).sum()}")
print(f"Genes with padj  < 0.10: {(res['padj']  < 0.10).sum()}")
print(f"\nTop 20 genes by pvalue:")
print(res.head(20)[["log2FoldChange", "pvalue", "padj"]].to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(res["pvalue"].dropna(), bins=50, color="steelblue", edgecolor="white")
axes[0].set_xlabel("pvalue")
axes[0].set_ylabel("Count")
axes[0].set_title("P-value distribution\n(enriched near 0 = real signal)")
axes[1].hist(res["padj"].dropna(), bins=50, color="salmon", edgecolor="white")
axes[1].set_xlabel("padj")
axes[1].set_title("Padj distribution")
plt.suptitle(f"{celltype_value} — {phase_to_test} | "
             f"{inj_types[0]} vs {inj_types[1]} (raw spatial counts)")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}pval_distribution_{analysis_label}.pdf", bbox_inches="tight")
plt.show()

# ══════════════════════════════════════════════════════════════════════════════
# 11. Volcano
# ══════════════════════════════════════════════════════════════════════════════
def volcano(df, analysis_label, inj_types, celltype_value, phase_to_test,
            design_str, use_padj=True, pval_thresh=0.05, lfc_thresh=0.58, top_n=10):

    pval_col   = "padj" if use_padj else "pvalue"
    pval_label = "padj" if use_padj else "p-value"

    df = df.dropna(subset=["log2FoldChange", pval_col]).copy()
    df["neglog10_pvalue"] = -np.log10(df["pvalue"].clip(lower=1e-300))

    sig_test = (df[pval_col] < pval_thresh) & (df["log2FoldChange"] >  lfc_thresh)
    sig_ref  = (df[pval_col] < pval_thresh) & (df["log2FoldChange"] < -lfc_thresh)
    other    = ~(sig_test | sig_ref)

    fig, ax = plt.subplots(figsize=(9, 7))

    ax.scatter(df.loc[other,    "log2FoldChange"], df.loc[other,    "neglog10_pvalue"],
               s=10, alpha=0.35, color="grey",      label="Not significant")
    ax.scatter(df.loc[sig_ref,  "log2FoldChange"], df.loc[sig_ref,  "neglog10_pvalue"],
               s=20, alpha=0.85, color="steelblue",
               label=f"Higher in {inj_types[0]} (n={sig_ref.sum()})")
    ax.scatter(df.loc[sig_test, "log2FoldChange"], df.loc[sig_test, "neglog10_pvalue"],
               s=20, alpha=0.85, color="orange",
               label=f"Higher in {inj_types[1]} (n={sig_test.sum()})")

    ax.axvline(-lfc_thresh, linestyle="--", linewidth=0.9, color="black", alpha=0.5)
    ax.axvline( lfc_thresh, linestyle="--", linewidth=0.9, color="black", alpha=0.5)

    if use_padj:
        sig_boundary = df[df[pval_col] < pval_thresh]["pvalue"]
        if len(sig_boundary) > 0:
            ax.axhline(-np.log10(sig_boundary.max()), linestyle="--",
                       linewidth=0.9, color="black", alpha=0.5)
    else:
        ax.axhline(-np.log10(pval_thresh), linestyle="--",
                   linewidth=0.9, color="black", alpha=0.5)

    texts = []
    for gene, row in df[sig_test].sort_values(pval_col).head(top_n).iterrows():
        texts.append(ax.text(row["log2FoldChange"], row["neglog10_pvalue"],
                             gene, fontsize=7.5))
    for gene, row in df[sig_ref].sort_values(pval_col).head(top_n).iterrows():
        texts.append(ax.text(row["log2FoldChange"], row["neglog10_pvalue"],
                             gene, fontsize=7.5))

    ax.set_xlabel(f"log₂ Fold Change ({inj_types[0]} vs {inj_types[1]})", fontsize=12)
    ax.set_ylabel("−log₁₀ p-value", fontsize=12)
    ax.set_title(
        f"{celltype_value} — {phase_to_test} (raw spatial counts)\n"
        f"{inj_types[0]} vs {inj_types[1]} | "
        f"{pval_label}<{pval_thresh}, |lfc|>{lfc_thresh} | design: {design_str}",
        fontsize=10
    )
    ax.legend(frameon=False)
    plt.tight_layout()
    save = f"{OUT_DIR}volcano_{analysis_label}.pdf"
    plt.savefig(save, bbox_inches="tight", dpi=150)
    plt.show()
    print(f"Volcano saved → {save}")

volcano(res, analysis_label, inj_types, celltype_value, phase_to_test,
        design_str, use_padj=use_padj, pval_thresh=pval_thresh,
        lfc_thresh=lfc_thresh, top_n=top_n_labels)